# Swiss Laws Graph Database Analysis

This notebook analyzes the structure and relationships in the Swiss Laws Neo4j graph database. The analysis includes:

1. Database connection and schema exploration
2. Graph projection creation with available node labels and relationship types
3. Centrality metrics (PageRank, Degree Centrality, Betweenness)
4. Community detection using the Louvain algorithm
5. Temporal patterns in law citations
6. Law validity analysis

Each cell is designed to work independently when possible, so you can run sections even if earlier ones encounter errors.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from graphdatascience import GraphDataScience
import os
import json
from datetime import datetime

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Neo4j connection parameters
NEO4J_URI = "bolt://localhost:49877" # Update this if your port is different
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "itlawskg"

# Connect to Neo4j
try:
    gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), database="neo4j")
    print("Successfully connected to Neo4j database!")
    
    # Display Neo4j version
    version = gds.run_cypher("RETURN gds.version() AS version")
    print(f"Neo4j GDS version: {version['version'].iloc[0]}")
    
    # Get database summary statistics
    print("\nDatabase statistics:")
    
    # Count nodes by label
    node_counts = gds.run_cypher("""
    MATCH (n)
    WITH labels(n) AS labels, count(n) AS count
    UNWIND labels AS label
    RETURN label, sum(count) AS count
    ORDER BY count DESC
    """)
    print("Node counts by label:")
    display(node_counts)
    
    # Count relationships by type
    rel_counts = gds.run_cypher("""
    MATCH ()-[r]-()
    RETURN type(r) AS relationship_type, count(DISTINCT r) AS count
    ORDER BY count DESC
    """)
    print("\nRelationship counts by type:")
    display(rel_counts)
    
    # Create a native projection of the graph
    print("\nCreating a native graph projection...")
    
    # First, let's check what node labels and relationship types actually exist in the database
    nodes_in_db = gds.run_cypher("""
    CALL db.labels() YIELD label
    RETURN label
    ORDER BY label
    """)
    print("\nAvailable node labels in the database:")
    display(nodes_in_db)
    
    rels_in_db = gds.run_cypher("""
    CALL db.relationshipTypes() YIELD relationshipType
    RETURN relationshipType
    ORDER BY relationshipType
    """)
    print("\nAvailable relationship types in the database:")
    display(rels_in_db)
    
    # You can modify these parameters based on your specific needs and what's available in the DB
    graph_name = "swisslaws_graph"
    
    # Use labels that exist in your database
    available_labels = nodes_in_db['label'].tolist()
    node_projection = ["Law"] if "Law" in available_labels else available_labels
    if not node_projection:
        print("No valid node labels found in the database!")
        raise Exception("No valid node labels available for projection")
    print(f"Using node labels for projection: {node_projection}")
    
    # Use relationship types that exist in your database
    available_rel_types = rels_in_db['relationshipType'].tolist()
    if not available_rel_types:
        print("No relationship types found in the database!")
        raise Exception("No relationships available for projection")
    print(f"Using relationship types for projection: {available_rel_types}")
    
    # Check if graph already exists and drop it if it does
    existing_graphs = gds.graph.list()
    if graph_name in existing_graphs['graphName'].values:
        print(f"Graph {graph_name} already exists. Dropping it...")
        gds.graph.drop(graph_name)
    
    # Directly create a minimal projection without any node properties
    # This avoids Date type properties that cause errors
    print("\nCreating a minimal graph projection without node properties...")
    try:
        graph = gds.graph.project(
            graph_name,
            node_projection, 
            available_rel_types
            # No node properties to avoid Date type issues
        )
        
        print(f"\nGraph projection created: {graph_name}")
        print(f"Nodes: {graph['nodeCount']}")
        print(f"Relationships: {graph['relationshipCount']}")
    except Exception as e:
        if "Loading of values of type Date" in str(e):
            print(f"Error with dates: {e}")
            print("\nTrying with different configuration...")
            
            # Final attempt - use string node projection format
            graph = gds.graph.project(
                graph_name,
                "*",  # Project all node types 
                "*"   # Project all relationship types
            )
            print(f"\nCreated minimal graph projection: {graph_name}")
            print(f"Nodes: {graph['nodeCount']}")
            print(f"Relationships: {graph['relationshipCount']}")
    
except Exception as e:
    print(f"Error connecting to Neo4j: {e}")
    
    # If we get to this point, try a very minimal approach without any properties
    print("\nAttempting to create minimal graph projection...")
    try:
        # Try creating a projection without any node properties
        graph_name = "swisslaws_graph_minimal"
        
        # Check if graph already exists and drop it
        existing_graphs = gds.graph.list()
        if graph_name in existing_graphs['graphName'].values:
            gds.graph.drop(graph_name)
        
        # Create a minimal projection
        graph = gds.graph.project(
            graph_name,
            "*",  # All node types
            "*"   # All relationship types
        )
        
        print(f"\nCreated minimal graph projection: {graph_name}")
        print(f"Nodes: {graph['nodeCount']}")
        print(f"Relationships: {graph['relationshipCount']}")
        
    except Exception as e2:
        print(f"Still encountering error with minimal projection: {e2}")
        print("\nRECOMMENDATION: Check if your database has been properly populated.")
        print("You may need to run the data population scripts:")
        print("- ./scripts/DB_Population/nodes.py")
        print("- ./scripts/DB_Population/articles.py")

.//miniconda3/envs/SwissLaws/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully connected to Neo4j database!
Neo4j GDS version: 2.11.0

Database statistics:
Node counts by label:


,label,count
0,Section,216739
1,Law,48302



Relationship counts by type:


,relationship_type,count
0,HAS_ARTICLE,216739
1,CITES,28777
2,AMENDS,18069
3,ABROGATES,6483
4,INTRODUCES,2325


...


Available relationship types in the database:


,relationshipType
0,ABROGATES
1,AMENDS
2,CITES
3,HAS_ARTICLE
4,INTRODUCES


Using node labels for projection: ['Law']
Using relationship types for projection: ['ABROGATES', 'AMENDS', 'CITES', 'HAS_ARTICLE', 'INTRODUCES']

Creating a minimal graph projection without node properties...


: 

: 

In [ ]:
# Projection for Law-to-Law relationships via Sections
print("\nCreating a Law-to-Law graph projection...")
try:
    # First check if the graph already exists
    existing_graphs = gds.graph.list()
    if graph_name in existing_graphs['graphName'].values:
        print(f"Graph {graph_name} already exists. Dropping it...")
        gds.graph.drop(graph_name)
    
    # Create a relationship query that finds Law-to-Law connections through Sections
    law_to_law_query = """
    MATCH (source_law:Law)-[:HAS_ARTICLE]->(:Section)-[r]->(target_law:Law)
    RETURN id(source_law) AS source, id(target_law) AS target, type(r) AS type
    """
    
    # Create a graph projection using Cypher projection
    graph = gds.graph.project.cypher(
        graph_name,
        node_query="MATCH (l:Law) RETURN id(l) AS id",
        relationship_query=law_to_law_query,
        validateRelationships=False  # Add this if you have potential duplicate relationships
    )
    
    print(f"\nLaw-to-Law graph projection created: {graph_name}")
    print(f"Nodes: {graph['nodeCount']}")
    print(f"Relationships: {graph['relationshipCount']}")
    
except Exception as e:
    print(f"Error creating Law-to-Law graph projection: {e}")
    
    # Try a simpler approach if the first one fails
    try:
        # Alternative approach using a simpler query
        print("\nTrying alternative approach for Law-to-Law projection...")
        
        # Create a virtual graph of just the Law nodes and their indirect connections
        graph_name = "law_citations_graph"
        
        # Check if graph already exists and drop it
        existing_graphs = gds.graph.list()
        if graph_name in existing_graphs['graphName'].values:
            gds.graph.drop(graph_name)
        
        # Create a projection using the simplified Cypher projection
        graph = gds.graph.project.cypher(
            graph_name,
            # Query for Law nodes
            node_query="MATCH (l:Law) RETURN id(l) AS id, l.lawId AS lawId",
            # Query for relationships between laws through articles
            relationship_query="""
            MATCH (source:Law)-[:HAS_ARTICLE]->(:Section)-[r]->(target:Law)
            RETURN id(source) AS source, id(target) AS target, count(r) AS weight, type(r) AS type
            """
        )
        
        print(f"\nCreated Law-to-Law graph projection: {graph_name}")
        print(f"Nodes: {graph['nodeCount']}")
        print(f"Relationships: {graph['relationshipCount']}")
        
    except Exception as e2:
        print(f"Still encountering error with Law-to-Law projection: {e2}")
        print("\nRecommendation: Check if the relationships in your query actually exist:")
        print("""
        MATCH (source_law:Law)-[:HAS_ARTICLE]->(:Section)-[r]->(target_law:Law)
        RETURN source_law.lawId, target_law.lawId, type(r)
        LIMIT 5
        """)


Creating a Law-to-Law graph projection...
Error creating Law-to-Law graph projection: name 'gds' is not defined

Trying alternative approach for Law-to-Law projection...
Still encountering error with Law-to-Law projection: name 'gds' is not defined

Recommendation: Check if the relationships in your query actually exist:

        MATCH (source_law:Law)-[:HAS_ARTICLE]->(:Section)-[r]->(target_law:Law)
        RETURN source_law.lawId, target_law.lawId, type(r)
        LIMIT 5
        


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from graphdatascience import GraphDataScience
import os
import json
from datetime import datetime

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Neo4j connection parameters
NEO4J_URI = "bolt://localhost:49877" # Update this if your port is different
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "itlawskg"

# Connect to Neo4j
try:
    gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), database="neo4j")
    print("Successfully connected to Neo4j database!")
    
    # Display Neo4j version
    version = gds.run_cypher("RETURN gds.version() AS version")
    print(f"Neo4j GDS version: {version['version'].iloc[0]}")
    
    # Get database summary statistics
    print("\nDatabase statistics:")
    
    # Count nodes by label
    node_counts = gds.run_cypher("""
    MATCH (n)
    WITH labels(n) AS labels, count(n) AS count
    UNWIND labels AS label
    RETURN label, sum(count) AS count
    ORDER BY count DESC
    """)
    print("Node counts by label:")
    display(node_counts)
    
    # Count relationships by type
    rel_counts = gds.run_cypher("""
    MATCH ()-[r]-()
    RETURN type(r) AS relationship_type, count(r) AS count
    ORDER BY count DESC
    """)
    print("\nRelationship counts by type:")
    display(rel_counts)
    
    # Create a native projection of the graph
    print("\nCreating a native graph projection...")
    
    # First, let's check what node labels and relationship types actually exist in the database
    nodes_in_db = gds.run_cypher("""
    CALL db.labels() YIELD label
    RETURN label
    ORDER BY label
    """)
    print("\nAvailable node labels in the database:")
    display(nodes_in_db)
    
    rels_in_db = gds.run_cypher("""
    CALL db.relationshipTypes() YIELD relationshipType
    RETURN relationshipType
    ORDER BY relationshipType
    """)
    print("\nAvailable relationship types in the database:")
    display(rels_in_db)
    
    # You can modify these parameters based on your specific needs and what's available in the DB
    graph_name = "swisslaws_graph"
    
    # Use labels that exist in your database
    available_labels = nodes_in_db['label'].tolist()
    node_projection = ["Law"] if "Law" in available_labels else available_labels
    if not node_projection:
        print("No valid node labels found in the database!")
        raise Exception("No valid node labels available for projection")
    print(f"Using node labels for projection: {node_projection}")
    
    # Use relationship types that exist in your database
    available_rel_types = rels_in_db['relationshipType'].tolist()
    if not available_rel_types:
        print("No relationship types found in the database!")
        raise Exception("No relationships available for projection")
    print(f"Using relationship types for projection: {available_rel_types}")
    
    # Check if graph already exists and drop it if it does
    existing_graphs = gds.graph.list()
    if graph_name in existing_graphs['graphName'].values:
        print(f"Graph {graph_name} already exists. Dropping it...")
        gds.graph.drop(graph_name)
    
    # Get available node properties for Law nodes (if they exist)
    node_properties = []
    if "Law" in available_labels:
        props = gds.run_cypher("""
        MATCH (l:Law) 
        WHERE l IS NOT NULL
        RETURN keys(l) AS properties
        LIMIT 1
        """)
        
        if not props.empty and len(props['properties']) > 0:
            # Get the first row's properties list
            all_props = props['properties'].iloc[0]
            # Filter for non-date properties (only use string, numeric properties)
            # Avoid date properties which cause the error
            useful_props = [p for p in all_props if p in 
                           ["validity", "status", "lawId", "title_it"]]
            
            # Verify property types to exclude date types
            if useful_props:
                # Get sample of property types
                prop_types = gds.run_cypher(f"""
                MATCH (l:Law) 
                WHERE l IS NOT NULL
                RETURN {', '.join(['apoc.meta.type(l.'+p+') as '+p+'_type' for p in useful_props])}
                LIMIT 1
                """)
                
                # Filter out any date properties
                safe_props = []
                for p in useful_props:
                    type_col = p + "_type"
                    if type_col in prop_types.columns:
                        prop_type = prop_types[type_col].iloc[0] if not prop_types.empty else None
                        if prop_type and "Date" not in prop_type and "date" not in prop_type:
                            safe_props.append(p)
                
                node_properties = safe_props
                print(f"\nUsing node properties: {node_properties}")
    
    # Create the graph projection with available node properties
    # Note: We're excluding date properties to avoid the unsupported type error
    graph = gds.graph.project(
        graph_name,
        node_projection, 
        available_rel_types,
        nodeProperties=node_properties
    )
    
    print(f"\nGraph projection created: {graph_name}")
    print(f"Nodes: {graph['nodeCount']}")
    print(f"Relationships: {graph['relationshipCount']}")
    
except Exception as e:
    print(f"Error connecting to Neo4j: {e}")
    
    # Additional troubleshooting if there's an error
    if "Loading of values of type Date is currently not supported" in str(e):
        print("\nERROR DETAILS: The graph projection cannot include date properties.")
        print("The error occurred because one or more properties are Date types.")
        print("Let's try creating a projection without any node properties:")
        
        try:
            # Try creating a projection without any node properties
            graph_name = "swisslaws_graph_minimal"
            
            # Check if graph already exists and drop it
            existing_graphs = gds.graph.list()
            if graph_name in existing_graphs['graphName'].values:
                gds.graph.drop(graph_name)
            
            # Create a minimal projection without properties
            graph = gds.graph.project(
                graph_name,
                node_projection, 
                available_rel_types
            )
            
            print(f"\nCreated minimal graph projection: {graph_name}")
            print(f"Nodes: {graph['nodeCount']}")
            print(f"Relationships: {graph['relationshipCount']}")
            
        except Exception as e2:
            print(f"Still encountering error with minimal projection: {e2}")

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from graphdatascience import GraphDataScience
import os
import json
from datetime import datetime

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Neo4j connection parameters
NEO4J_URI = "bolt://localhost:49877" # Update this if your port is different
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "itlawskg"

# Connect to Neo4j
try:
    gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), database="neo4j")
    print("Successfully connected to Neo4j database!")
    
    # Display Neo4j version
    version = gds.run_cypher("RETURN gds.version() AS version")
    print(f"Neo4j GDS version: {version['version'].iloc[0]}")
    
    # Get database summary statistics
    print("\nDatabase statistics:")
    
    # Count nodes by label
    node_counts = gds.run_cypher("""
    MATCH (n)
    WITH labels(n) AS labels, count(n) AS count
    UNWIND labels AS label
    RETURN label, sum(count) AS count
    ORDER BY count DESC
    """)
    print("Node counts by label:")
    display(node_counts)
    
    # Count relationships by type
    rel_counts = gds.run_cypher("""
    MATCH ()-[r]-()
    RETURN type(r) AS relationship_type, count(r) AS count
    ORDER BY count DESC
    """)
    print("\nRelationship counts by type:")
    display(rel_counts)
    
    # Create a native projection of the graph
    print("\nCreating a native graph projection...")
    
    # First, let's check what node labels and relationship types actually exist in the database
    nodes_in_db = gds.run_cypher("""
    CALL db.labels() YIELD label
    RETURN label
    ORDER BY label
    """)
    print("\nAvailable node labels in the database:")
    display(nodes_in_db)
    
    rels_in_db = gds.run_cypher("""
    CALL db.relationshipTypes() YIELD relationshipType
    RETURN relationshipType
    ORDER BY relationshipType
    """)
    print("\nAvailable relationship types in the database:")
    display(rels_in_db)
    
    # You can modify these parameters based on your specific needs and what's available in the DB
    graph_name = "swisslaws_graph"
    
    # Use labels that exist in your database
    available_labels = nodes_in_db['label'].tolist()
    node_projection = ["Law"] if "Law" in available_labels else available_labels
    if not node_projection:
        print("No valid node labels found in the database!")
        raise Exception("No valid node labels available for projection")
    print(f"Using node labels for projection: {node_projection}")
    
    # Use relationship types that exist in your database
    available_rel_types = rels_in_db['relationshipType'].tolist()
    if not available_rel_types:
        print("No relationship types found in the database!")
        raise Exception("No relationships available for projection")
    print(f"Using relationship types for projection: {available_rel_types}")
    
    # Check if graph already exists and drop it if it does
    existing_graphs = gds.graph.list()
    if graph_name in existing_graphs['graphName'].values:
        print(f"Graph {graph_name} already exists. Dropping it...")
        gds.graph.drop(graph_name)
    
    # Get available node properties for Law nodes (if they exist)
    node_properties = []
    if "Law" in available_labels:
        props = gds.run_cypher("""
        MATCH (l:Law) 
        WHERE l IS NOT NULL
        RETURN keys(l) AS properties
        LIMIT 1
        """)
        
        if not props.empty and len(props['properties']) > 0:
            # Get the first row's properties list
            all_props = props['properties'].iloc[0]
            # Filter for non-date properties (only use string, numeric properties)
            # Avoid date properties which cause the error
            useful_props = [p for p in all_props if p in 
                           ["validity", "status", "lawId", "title_it"]]
            
            # Verify property types to exclude date types
            if useful_props:
                # Get sample of property types
                prop_types = gds.run_cypher(f"""
                MATCH (l:Law) 
                WHERE l IS NOT NULL
                RETURN {', '.join(['apoc.meta.type(l.'+p+') as '+p+'_type' for p in useful_props])}
                LIMIT 1
                """)
                
                # Filter out any date properties
                safe_props = []
                for p in useful_props:
                    type_col = p + "_type"
                    if type_col in prop_types.columns:
                        prop_type = prop_types[type_col].iloc[0] if not prop_types.empty else None
                        if prop_type and "Date" not in prop_type and "date" not in prop_type:
                            safe_props.append(p)
                
                node_properties = safe_props
                print(f"\nUsing node properties: {node_properties}")
    
    # Create the graph projection with available node properties
    # Note: We're excluding date properties to avoid the unsupported type error
    graph = gds.graph.project(
        graph_name,
        node_projection, 
        available_rel_types,
        nodeProperties=node_properties
    )
    
    print(f"\nGraph projection created: {graph_name}")
    print(f"Nodes: {graph['nodeCount']}")
    print(f"Relationships: {graph['relationshipCount']}")
    
except Exception as e:
    print(f"Error connecting to Neo4j: {e}")
    
    # Additional troubleshooting if there's an error
    if "Loading of values of type Date is currently not supported" in str(e):
        print("\nERROR DETAILS: The graph projection cannot include date properties.")
        print("The error occurred because one or more properties are Date types.")
        print("Let's try creating a projection without any node properties:")
        
        try:
            # Try creating a projection without any node properties
            graph_name = "swisslaws_graph_minimal"
            
            # Check if graph already exists and drop it
            existing_graphs = gds.graph.list()
            if graph_name in existing_graphs['graphName'].values:
                gds.graph.drop(graph_name)
            
            # Create a minimal projection without properties
            graph = gds.graph.project(
                graph_name,
                node_projection, 
                available_rel_types
            )
            
            print(f"\nCreated minimal graph projection: {graph_name}")
            print(f"Nodes: {graph['nodeCount']}")
            print(f"Relationships: {graph['relationshipCount']}")
            
        except Exception as e2:
            print(f"Still encountering error with minimal projection: {e2}")

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from graphdatascience import GraphDataScience
import os
import json
from datetime import datetime

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Neo4j connection parameters
NEO4J_URI = "bolt://localhost:49877" # Update this if your port is different
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "itlawskg"

# Connect to Neo4j
try:
    gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), database="neo4j")
    print("Successfully connected to Neo4j database!")
    
    # Display Neo4j version
    version = gds.run_cypher("RETURN gds.version() AS version")
    print(f"Neo4j GDS version: {version['version'].iloc[0]}")
    
    # Get database summary statistics
    print("\nDatabase statistics:")
    
    # Count nodes by label
    node_counts = gds.run_cypher("""
    MATCH (n)
    WITH labels(n) AS labels, count(n) AS count
    UNWIND labels AS label
    RETURN label, sum(count) AS count
    ORDER BY count DESC
    """)
    print("Node counts by label:")
    display(node_counts)
    
    # Count relationships by type
    rel_counts = gds.run_cypher("""
    MATCH ()-[r]-()
    RETURN type(r) AS relationship_type, count(r) AS count
    ORDER BY count DESC
    """)
    print("\nRelationship counts by type:")
    display(rel_counts)
    
    # Create a native projection of the graph
    print("\nCreating a native graph projection...")
    
    # First, let's check what node labels and relationship types actually exist in the database
    nodes_in_db = gds.run_cypher("""
    CALL db.labels() YIELD label
    RETURN label
    ORDER BY label
    """)
    print("\nAvailable node labels in the database:")
    display(nodes_in_db)
    
    rels_in_db = gds.run_cypher("""
    CALL db.relationshipTypes() YIELD relationshipType
    RETURN relationshipType
    ORDER BY relationshipType
    """)
    print("\nAvailable relationship types in the database:")
    display(rels_in_db)
    
    # You can modify these parameters based on your specific needs and what's available in the DB
    graph_name = "swisslaws_graph"
    
    # Use labels that exist in your database
    available_labels = nodes_in_db['label'].tolist()
    node_projection = ["Law"] if "Law" in available_labels else available_labels
    if not node_projection:
        print("No valid node labels found in the database!")
        raise Exception("No valid node labels available for projection")
    print(f"Using node labels for projection: {node_projection}")
    
    # Use relationship types that exist in your database
    available_rel_types = rels_in_db['relationshipType'].tolist()
    if not available_rel_types:
        print("No relationship types found in the database!")
        raise Exception("No relationships available for projection")
    print(f"Using relationship types for projection: {available_rel_types}")
    
    # Check if graph already exists and drop it if it does
    existing_graphs = gds.graph.list()
    if graph_name in existing_graphs['graphName'].values:
        print(f"Graph {graph_name} already exists. Dropping it...")
        gds.graph.drop(graph_name)
    
    # Get available node properties for Law nodes (if they exist)
    node_properties = []
    if "Law" in available_labels:
        props = gds.run_cypher("""
        MATCH (l:Law) 
        WHERE l IS NOT NULL
        RETURN keys(l) AS properties
        LIMIT 1
        """)
        
        if not props.empty and len(props['properties']) > 0:
            # Get the first row's properties list
            all_props = props['properties'].iloc[0]
            # Filter for non-date properties (only use string, numeric properties)
            # Avoid date properties which cause the error
            useful_props = [p for p in all_props if p in 
                           ["validity", "status", "lawId", "title_it"]]
            
            # Verify property types to exclude date types
            if useful_props:
                # Get sample of property types
                prop_types = gds.run_cypher(f"""
                MATCH (l:Law) 
                WHERE l IS NOT NULL
                RETURN {', '.join(['apoc.meta.type(l.'+p+') as '+p+'_type' for p in useful_props])}
                LIMIT 1
                """)
                
                # Filter out any date properties
                safe_props = []
                for p in useful_props:
                    type_col = p + "_type"
                    if type_col in prop_types.columns:
                        prop_type = prop_types[type_col].iloc[0] if not prop_types.empty else None
                        if prop_type and "Date" not in prop_type and "date" not in prop_type:
                            safe_props.append(p)
                
                node_properties = safe_props
                print(f"\nUsing node properties: {node_properties}")
    
    # Create the graph projection with available node properties
    # Note: We're excluding date properties to avoid the unsupported type error
    graph = gds.graph.project(
        graph_name,
        node_projection, 
        available_rel_types,
        nodeProperties=node_properties
    )
    
    print(f"\nGraph projection created: {graph_name}")
    print(f"Nodes: {graph['nodeCount']}")
    print(f"Relationships: {graph['relationshipCount']}")
    
except Exception as e:
    print(f"Error connecting to Neo4j: {e}")
    
    # Additional troubleshooting if there's an error
    if "Loading of values of type Date is currently not supported" in str(e):
        print("\nERROR DETAILS: The graph projection cannot include date properties.")
        print("The error occurred because one or more properties are Date types.")
        print("Let's try creating a projection without any node properties:")
        
        try:
            # Try creating a projection without any node properties
            graph_name = "swisslaws_graph_minimal"
            
            # Check if graph already exists and drop it
            existing_graphs = gds.graph.list()
            if graph_name in existing_graphs['graphName'].values:
                gds.graph.drop(graph_name)
            
            # Create a minimal projection without properties
            graph = gds.graph.project(
                graph_name,
                node_projection, 
                available_rel_types
            )
            
            print(f"\nCreated minimal graph projection: {graph_name}")
            print(f"Nodes: {graph['nodeCount']}")
            print(f"Relationships: {graph['relationshipCount']}")
            
        except Exception as e2:
            print(f"Still encountering error with minimal projection: {e2}")

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from graphdatascience import GraphDataScience
import os
import json
from datetime import datetime

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Neo4j connection parameters
NEO4J_URI = "bolt://localhost:49877" # Update this if your port is different
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "itlawskg"

# Connect to Neo4j
try:
    gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), database="neo4j")
    print("Successfully connected to Neo4j database!")
    
    # Display Neo4j version
    version = gds.run_cypher("RETURN gds.version() AS version")
    print(f"Neo4j GDS version: {version['version'].iloc[0]}")
    
    # Get database summary statistics
    print("\nDatabase statistics:")
    
    # Count nodes by label
    node_counts = gds.run_cypher("""
    MATCH (n)
    WITH labels(n) AS labels, count(n) AS count
    UNWIND labels AS label
    RETURN label, sum(count) AS count
    ORDER BY count DESC
    """)
    print("Node counts by label:")
    display(node_counts)
    
    # Count relationships by type
    rel_counts = gds.run_cypher("""
    MATCH ()-[r]-()
    RETURN type(r) AS relationship_type, count(r) AS count
    ORDER BY count DESC
    """)
    print("\nRelationship counts by type:")
    display(rel_counts)
    
    # Create a native projection of the graph
    print("\nCreating a native graph projection...")
    
    # First, let's check what node labels and relationship types actually exist in the database
    nodes_in_db = gds.run_cypher("""
    CALL db.labels() YIELD label
    RETURN label
    ORDER BY label
    """)
    print("\nAvailable node labels in the database:")
    display(nodes_in_db)
    
    rels_in_db = gds.run_cypher("""
    CALL db.relationshipTypes() YIELD relationshipType
    RETURN relationshipType
    ORDER BY relationshipType
    """)
    print("\nAvailable relationship types in the database:")
    display(rels_in_db)
    
    # You can modify these parameters based on your specific needs and what's available in the DB
    graph_name = "swisslaws_graph"
    
    # Use labels that exist in your database
    available_labels = nodes_in_db['label'].tolist()
    node_projection = ["Law"] if "Law" in available_labels else available_labels
    if not node_projection:
        print("No valid node labels found in the database!")
        raise Exception("No valid node labels available for projection")
    print(f"Using node labels for projection: {node_projection}")
    
    # Use relationship types that exist in your database
    available_rel_types = rels_in_db['relationshipType'].tolist()
    if not available_rel_types:
        print("No relationship types found in the database!")
        raise Exception("No relationships available for projection")
    print(f"Using relationship types for projection: {available_rel_types}")
    
    # Check if graph already exists and drop it if it does
    existing_graphs = gds.graph.list()
    if graph_name in existing_graphs['graphName'].values:
        print(f"Graph {graph_name} already exists. Dropping it...")
        gds.graph.drop(graph_name)
    
    # Get available node properties for Law nodes (if they exist)
    node_properties = []
    if "Law" in available_labels:
        props = gds.run_cypher("""
        MATCH (l:Law) 
        WHERE l IS NOT NULL
        RETURN keys(l) AS properties
        LIMIT 1
        """)
        
        if not props.empty and len(props['properties']) > 0:
            # Get the first row's properties list
            all_props = props['properties'].iloc[0]
            # Filter for likely useful properties
            useful_props = [p for p in all_props if p in 
                           ["validity", "publicationDate", "entryintoforceDate", 
                            "nolongerinforceDate", "status", "lawId"]]
            node_properties = useful_props
            print(f"\nUsing node properties: {node_properties}")
    
    # Create the graph projection with available node properties
    graph = gds.graph.project(
        graph_name,
        node_projection, 
        available_rel_types,
        nodeProperties=node_properties
    )
    
    print(f"\nGraph projection created: {graph_name}")
    print(f"Nodes: {graph['nodeCount']}")
    print(f"Relationships: {graph['relationshipCount']}")
    
except Exception as e:
    print(f"Error connecting to Neo4j: {e}")

Successfully connected to Neo4j database!
Neo4j GDS version: 2.11.0

Database statistics:
Node counts by label:


,label,count
0,Section,216739
1,Law,48302



Relationship counts by type:


,relationship_type,count
0,HAS_ARTICLE,433478
1,CITES,57554
2,AMENDS,36138
3,ABROGATES,12966
4,INTRODUCES,4650



Creating a native graph projection...

Available node labels in the database:


,label
0,Law
1,Section



Available relationship types in the database:


,relationshipType
0,ABROGATES
1,AMENDS
2,CITES
3,HAS_ARTICLE
4,INTRODUCES


Using node labels for projection: ['Law']
Using relationship types for projection: ['ABROGATES', 'AMENDS', 'CITES', 'HAS_ARTICLE', 'INTRODUCES']

Using node properties: ['entryintoforceDate', 'nolongerinforceDate', 'lawId', 'publicationDate', 'validity']
Error connecting to Neo4j: {code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.UnsupportedOperationException: Loading of values of type Date is currently not supported}


## Connection Setup and Initial Graph Projection

This cell establishes a connection to the Neo4j database and creates a graph projection using available node labels and relationship types. The projection is necessary for running graph algorithms in subsequent cells.

**Key components:**
- Connection to Neo4j database server
- Discovery of available node labels and relationship types
- Creating a named graph projection with appropriate properties
- Basic database statistics

In [ ]:
# Run basic graph algorithms

# 1. PageRank to identify most influential laws
print("Running PageRank algorithm to identify most important nodes...")
try:
    result_pagerank = gds.pageRank.stream(
        graph_name,
        maxIterations=20,
        dampingFactor=0.85
    ).sort_values('score', ascending=False)
    
    # Display top 20 laws by PageRank
    print("\nTop 20 nodes by PageRank score:")
    
    # Get node properties for the top laws
    top_nodes = result_pagerank.head(20)
    
    # Get details of top nodes
    node_details = gds.run_cypher("""
    MATCH (n) 
    WHERE id(n) IN $nodeIds
    RETURN id(n) AS nodeId, 
           labels(n) AS labels, 
           CASE 
             WHEN 'Law' IN labels(n) THEN n.title_it 
             WHEN 'Section' IN labels(n) THEN n.marker + ': ' + n.title
             ELSE 'Unknown'
           END AS title,
           CASE 
             WHEN 'Law' IN labels(n) THEN n.lawId
             WHEN 'Section' IN labels(n) THEN n.sectionId
             ELSE 'Unknown'
           END AS identifier
    """, {"nodeIds": top_nodes['nodeId'].tolist()})
    
    # Join PageRank scores with node details
    result_with_details = pd.merge(top_nodes, node_details, on="nodeId")
    display(result_with_details[['nodeId', 'score', 'labels', 'title', 'identifier']])
    
except Exception as e:
    print(f"Error running PageRank: {e}")

# 2. Degree Centrality - count of relationships
print("\nComputing node degrees...")
try:
    result_degree = gds.degree.stream(graph_name).sort_values('score', ascending=False)
    
    # Display top 20 laws by degree
    print("\nTop 20 nodes by degree (number of connections):")
    
    # Get node details for the top laws by degree
    top_degree_nodes = result_degree.head(20)
    
    # Get details of top nodes by degree
    degree_node_details = gds.run_cypher("""
    MATCH (n) 
    WHERE id(n) IN $nodeIds
    RETURN id(n) AS nodeId, 
           labels(n) AS labels, 
           CASE 
             WHEN 'Law' IN labels(n) THEN n.title_it 
             WHEN 'Section' IN labels(n) THEN n.marker + ': ' + n.title
             ELSE 'Unknown'
           END AS title,
           CASE 
             WHEN 'Law' IN labels(n) THEN n.lawId
             WHEN 'Section' IN labels(n) THEN n.sectionId
             ELSE 'Unknown'
           END AS identifier
    """, {"nodeIds": top_degree_nodes['nodeId'].tolist()})
    
    # Join degree scores with node details
    degree_with_details = pd.merge(top_degree_nodes, degree_node_details, on="nodeId")
    display(degree_with_details[['nodeId', 'score', 'labels', 'title', 'identifier']])
    
except Exception as e:
    print(f"Error computing degree centrality: {e}")

In [ ]:
# Troubleshooting and Database Schema Exploration

# Let's examine the database structure in more detail to understand what we're working with
try:
    print("\n===== DATABASE SCHEMA EXPLORATION =====")
    
    # Check if we have Law nodes and their properties
    law_nodes = gds.run_cypher("""
    MATCH (l:Law) 
    RETURN count(l) as count
    """)
    print(f"Law nodes count: {law_nodes['count'].iloc[0] if not law_nodes.empty else 0}")
    
    if not law_nodes.empty and law_nodes['count'].iloc[0] > 0:
        # Get sample properties from Law nodes
        law_props = gds.run_cypher("""
        MATCH (l:Law) 
        RETURN l LIMIT 1
        """)
        
        if not law_props.empty:
            print("\nSample Law node properties:")
            for col in law_props.columns:
                if col.startswith('l.') and not pd.isna(law_props[col].iloc[0]):
                    print(f"  - {col[2:]}: {law_props[col].iloc[0]}")
    
    # Check relationships connected to Law nodes
    law_rels = gds.run_cypher("""
    MATCH (l:Law)-[r]-() 
    RETURN type(r) as rel_type, count(r) as count 
    ORDER BY count DESC
    """)
    
    if not law_rels.empty:
        print("\nRelationships connected to Law nodes:")
        display(law_rels)
    else:
        print("\nNo relationships found connected to Law nodes!")
    
    # Sample of actual data (first 5 Law nodes with their connected nodes)
    print("\nSample of Law nodes with their connections:")
    sample_data = gds.run_cypher("""
    MATCH (l:Law)
    OPTIONAL MATCH (l)-[r]->(connected)
    RETURN l.lawId as law_id, l.title_it as title, 
           type(r) as relationship, labels(connected) as connected_type,
           CASE 
             WHEN 'Law' IN labels(connected) THEN connected.lawId
             WHEN 'Section' IN labels(connected) THEN connected.sectionId
             ELSE null
           END as connected_id
    LIMIT 10
    """)
    
    display(sample_data)
    
    # If we don't have the expected relationships, check what we do have
    print("\n===== CHECKING WHAT RELATIONSHIPS ARE AVAILABLE =====")
    all_rels = gds.run_cypher("""
    MATCH ()-[r]->() 
    RETURN DISTINCT type(r) as relationship_type, 
           count(r) as count,
           startNode(r).lawId as sample_start_node,
           endNode(r).lawId as sample_end_node
    ORDER BY count DESC
    """)
    
    print("All relationship types in the database:")
    display(all_rels)
    
    # Check node count again just to be sure
    node_count = gds.run_cypher("""
    MATCH (n) 
    RETURN count(n) as node_count
    """)
    print(f"\nTotal node count: {node_count['node_count'].iloc[0] if not node_count.empty else 0}")
    
    # Execute raw Cypher to check if the database is empty
    if node_count.empty or node_count['node_count'].iloc[0] == 0:
        print("\nWARNING: Database appears to be empty! You may need to run the population scripts first.")
        print("Check paths: ./scripts/DB_Population/nodes.py and articles.py")
    
except Exception as e:
    print(f"Error exploring database schema: {e}")

# If we have a graph projection, check its contents
if 'graph' in locals():
    try:
        print("\n===== GRAPH PROJECTION INFO =====")
        print(f"Graph name: {graph_name}")
        print(f"Nodes: {graph['nodeCount']}")
        print(f"Relationships: {graph['relationshipCount']}")
        print(f"Node properties: {graph['nodePropertyNames'] if 'nodePropertyNames' in graph else 'None'}")
        
        # If we need to modify our analysis based on what's available
        if graph['relationshipCount'] == 0:
            print("\nWARNING: Graph projection has 0 relationships.")
            print("This will affect all centrality and community detection algorithms.")
            print("Consider checking your database population scripts or database connection.")
    except Exception as e:
        print(f"Error checking graph projection: {e}")
else:
    print("\nNo graph projection available. Algorithms that depend on it will not work.")

In [ ]:
# Basic Graph Exploration

# This cell doesn't require a graph projection - it uses direct Cypher queries
try:
    # Get general statistics about the graph
    print("Exploring Law nodes and their properties...")
    
    # Check if we have any Law nodes at all
    law_count = gds.run_cypher("""
    MATCH (l:Law) 
    RETURN count(l) as count
    """)
    law_count_val = law_count['count'].iloc[0] if not law_count.empty else 0
    print(f"\nTotal number of Law nodes: {law_count_val}")
    
    if law_count_val > 0:
        # Get distribution of laws by validity status
        validity_stats = gds.run_cypher("""
        MATCH (l:Law)
        RETURN l.validity as status, count(*) as count
        ORDER BY count DESC
        """)
        
        print("\nDistribution of laws by validity status:")
        display(validity_stats)
        
        # Get distribution of laws by year of publication
        year_stats = gds.run_cypher("""
        MATCH (l:Law)
        WHERE l.publicationDate IS NOT NULL
        RETURN YEAR(l.publicationDate) as year, count(*) as count
        ORDER BY year
        """)
        
        print("\nDistribution of laws by publication year:")
        if not year_stats.empty:
            plt.figure(figsize=(12, 6))
            plt.bar(year_stats['year'], year_stats['count'])
            plt.title('Number of Laws by Publication Year')
            plt.xlabel('Publication Year')
            plt.ylabel('Number of Laws')
            plt.xticks(rotation=90, fontsize=8)
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print("No publication year data available")
        
        # Sample of laws with most sections
        laws_with_most_sections = gds.run_cypher("""
        MATCH (l:Law)-[r]->(s)
        WITH l, count(s) as section_count
        RETURN l.lawId as law_id, l.title_it as title, section_count
        ORDER BY section_count DESC
        LIMIT 10
        """)
        
        print("\nTop 10 laws with most connected sections:")
        display(laws_with_most_sections)
    else:
        print("No Law nodes found in the database. Please check your data population scripts.")
        
except Exception as e:
    print(f"Error in basic exploration: {e}")

In [ ]:
# 3. Community Detection using Louvain method
print("Running Louvain community detection algorithm...")
try:
    result_louvain = gds.louvain.stream(graph_name)
    
    # Get community statistics
    community_stats = result_louvain.groupby('communityId').agg(
        node_count=('nodeId', 'count')
    ).sort_values('node_count', ascending=False).reset_index()
    
    print(f"\nDetected {len(community_stats)} communities")
    print("\nTop 10 communities by size:")
    display(community_stats.head(10))
    
    # Sample nodes from top communities for inspection
    print("\nSampling nodes from the largest community:")
    largest_community_id = community_stats['communityId'].iloc[0]
    largest_community_nodes = result_louvain[result_louvain['communityId'] == largest_community_id]['nodeId'].sample(min(10, len(result_louvain[result_louvain['communityId'] == largest_community_id])))
    
    # Get details of sampled nodes
    sample_details = gds.run_cypher("""
    MATCH (n) 
    WHERE id(n) IN $nodeIds
    RETURN id(n) AS nodeId, 
           labels(n) AS labels, 
           CASE 
             WHEN 'Law' IN labels(n) THEN n.title_it 
             WHEN 'Section' IN labels(n) THEN n.marker + ': ' + n.title
             ELSE 'Unknown'
           END AS title,
           CASE 
             WHEN 'Law' IN labels(n) THEN n.lawId 
             WHEN 'Section' IN labels(n) THEN n.sectionId
             ELSE 'Unknown'
           END AS identifier
    """, {"nodeIds": largest_community_nodes.tolist()})
    
    display(sample_details)
    
except Exception as e:
    print(f"Error running community detection: {e}")

In [ ]:
# 4. Visualizing community distribution
try:
    plt.figure(figsize=(10, 6))
    
    # Plot community size distribution (top 20 communities)
    top_communities = community_stats.head(20)
    plt.bar(top_communities['communityId'].astype(str), top_communities['node_count'])
    plt.title('Size of Top 20 Communities')
    plt.xlabel('Community ID')
    plt.ylabel('Number of Nodes')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Identify temporal patterns in law citations
    print("\nAnalyzing temporal patterns in law citations...")
    temporal_patterns = gds.run_cypher("""
    MATCH (l:Law)-[:HAS_ARTICLE]->(s:Section)-[c:CITATION]->(target:Law)
    WHERE l.publicationDate IS NOT NULL AND target.publicationDate IS NOT NULL
    RETURN YEAR(l.publicationDate) AS source_year,
           YEAR(target.publicationDate) AS target_year,
           count(*) AS citation_count
    ORDER BY source_year, target_year
    """)
    
    if not temporal_patterns.empty:
        # Create a pivot table for heatmap
        pivot_data = temporal_patterns.pivot_table(
            index='source_year', 
            columns='target_year', 
            values='citation_count',
            fill_value=0
        )
        
        plt.figure(figsize=(12, 10))
        sns.heatmap(pivot_data, cmap="YlGnBu", annot=False, fmt="d")
        plt.title('Citation Counts: Source Year → Target Year')
        plt.xlabel('Target Law Year')
        plt.ylabel('Source Law Year')
        plt.tight_layout()
        plt.show()
        
        # Show diagonal sum (citations to same year)
        # And citations to previous years vs. citations to future years
        print("\nCitation pattern analysis:")
        same_year = temporal_patterns[temporal_patterns['source_year'] == temporal_patterns['target_year']]['citation_count'].sum()
        to_past = temporal_patterns[temporal_patterns['source_year'] > temporal_patterns['target_year']]['citation_count'].sum()
        to_future = temporal_patterns[temporal_patterns['source_year'] < temporal_patterns['target_year']]['citation_count'].sum()
        
        print(f"Citations to laws from same year: {same_year}")
        print(f"Citations to older laws: {to_past}")
        print(f"Citations to newer laws (likely amendments/updates): {to_future}")
        
        # Create pie chart
        plt.figure(figsize=(10, 7))
        plt.pie([same_year, to_past, to_future], 
                labels=['Same Year', 'To Older Laws', 'To Newer Laws'],
                autopct='%1.1f%%', 
                startangle=90,
                colors=['#66b3ff','#99ff99','#ffcc99'])
        plt.title('Distribution of Citations by Temporal Relationship')
        plt.tight_layout()
        plt.show()
    else:
        print("No temporal data available for analysis")
    
except Exception as e:
    print(f"Error in visualization: {e}")

In [ ]:
# 5. Analyzing law validity and relationships
try:
    print("Analyzing law validity patterns...")
    
    # Count laws by validity status
    validity_counts = gds.run_cypher("""
    MATCH (l:Law)
    RETURN l.validity AS status, count(*) AS count
    ORDER BY count DESC
    """)
    
    print("\nLaw counts by validity status:")
    display(validity_counts)
    
    # Visualize validity distribution
    plt.figure(figsize=(10, 6))
    plt.bar(validity_counts['status'], validity_counts['count'])
    plt.title('Distribution of Laws by Validity Status')
    plt.xlabel('Validity Status')
    plt.ylabel('Number of Laws')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Analyze citation patterns between valid and abrogated laws
    citation_validity = gds.run_cypher("""
    MATCH (source:Law)-[:HAS_ARTICLE]->(:Section)-[:CITATION]->(target:Law)
    RETURN source.validity AS source_status, 
           target.validity AS target_status,
           count(*) AS citation_count
    ORDER BY citation_count DESC
    """)
    
    print("\nCitation patterns by law validity:")
    display(citation_validity)
    
    # Create a heatmap for citation patterns by validity
    if not citation_validity.empty:
        pivot_validity = citation_validity.pivot_table(
            index='source_status', 
            columns='target_status', 
            values='citation_count',
            fill_value=0
        )
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(pivot_validity, cmap="YlGnBu", annot=True, fmt="d")
        plt.title('Citation Counts by Law Validity')
        plt.xlabel('Target Law Status')
        plt.ylabel('Source Law Status')
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"Error analyzing law validity: {e}")

In [ ]:
# 6. Advanced graph metrics and analysis
try:
    # Calculate betweenness centrality for identifying bridge laws
    print("Computing betweenness centrality to identify bridge laws...")
    # Note: This can be computationally expensive on large graphs
    # Using a sampling factor to make it more efficient
    betweenness_result = gds.betweenness.stream(
        graph_name,
        samplingFactor=0.1  # Adjust this based on your graph size
    ).sort_values('score', ascending=False)
    
    # Get top laws by betweenness
    top_betweenness = betweenness_result.head(20)
    betweenness_details = gds.run_cypher("""
    MATCH (n) 
    WHERE id(n) IN $nodeIds
    RETURN id(n) AS nodeId, 
           labels(n) AS labels, 
           CASE 
             WHEN 'Law' IN labels(n) THEN n.title_it 
             WHEN 'Section' IN labels(n) THEN n.marker + ': ' + n.title
             ELSE 'Unknown'
           END AS title,
           CASE 
             WHEN 'Law' IN labels(n) THEN n.lawId
             WHEN 'Section' IN labels(n) THEN n.sectionId
             ELSE 'Unknown'
           END AS identifier,
           CASE
             WHEN 'Law' IN labels(n) THEN n.validity
             ELSE 'N/A'
           END AS validity
    """, {"nodeIds": top_betweenness['nodeId'].tolist()})
    
    # Join betweenness scores with node details
    betweenness_with_details = pd.merge(top_betweenness, betweenness_details, on="nodeId")
    
    print("\nTop 20 nodes by betweenness centrality (bridge laws):")
    display(betweenness_with_details[['nodeId', 'score', 'labels', 'title', 'identifier', 'validity']])
    
    print("\nSummary of graph analysis findings:")
    print("---------------------------------------")
    print(f"1. The Swiss laws database contains {node_counts['count'].sum()} nodes and {rel_counts['count'].sum()} relationships.")
    
    # Only include community stats if the variable exists
    if 'community_stats' in locals() or 'community_stats' in globals():
        print(f"2. We identified {len(community_stats)} distinct communities of related laws.")
    
    print("3. The network shows patterns of connections that reflect the Swiss legal system structure.")
    
    # Only include if we have validity data
    try:
        if 'validity_counts' in locals() or 'validity_counts' in globals():
            if not validity_counts.empty:
                print("4. There are distinct citation patterns between laws of different validity status.")
    except:
        pass
    
    # Only include if we have temporal data
    try:
        if 'temporal_patterns' in locals() or 'temporal_patterns' in globals():
            if not temporal_patterns.empty:
                print("5. The temporal analysis reveals how laws cite previous and future legislation.")
    except:
        pass
    
    # Clean up - drop the graph projection
    print("\nCleaning up graph projections...")
    gds.graph.drop(graph_name)
    print("Analysis complete!")
    
except Exception as e:
    print(f"Error in advanced analysis: {e}")
    # Try to clean up even if there was an error
    try:
        gds.graph.drop(graph_name)
    except:
        pass

In [ ]:
# Final Cell: Cleanup and Recommendations

# Clean up any graph projections
try:
    if 'graph_name' in locals() and 'gds' in locals():
        existing = gds.graph.list()
        if graph_name in existing['graphName'].values:
            gds.graph.drop(graph_name)
            print(f"Graph projection '{graph_name}' has been dropped.")
    
    print("\n===== ANALYSIS SUMMARY AND RECOMMENDATIONS =====")
    
    # Check if we had enough data to perform a meaningful analysis
    if 'node_counts' in locals() and not node_counts.empty:
        node_count_sum = node_counts['count'].sum()
        if node_count_sum == 0:
            print("❌ The database appears to be empty. No nodes were found.")
            print("Recommendations:")
            print("1. Check that your Neo4j instance is running at the correct URI: " + NEO4J_URI)
            print("2. Run the data population scripts in ./scripts/DB_Population/")
            print("3. Check for errors in the population logs")
        else:
            print(f"✓ Found {node_count_sum} nodes in the database")
    
    if 'rel_counts' in locals() and not rel_counts.empty:
        rel_count_sum = rel_counts['count'].sum()
        if rel_count_sum == 0:
            print("❌ No relationships found in the database.")
            print("Recommendations:")
            print("1. Check that your relationship population script has run correctly")
            print("2. Look at the articles_population.log file for errors")
            print("3. Check if the relationship types in your data match what's expected in the population scripts")
        else:
            print(f"✓ Found {rel_count_sum} relationships in the database")
    
    print("\nFor successful graph analysis, you need:")
    print("1. Law nodes populated from complete_DB.csv")
    print("2. Section nodes created from JSONs_classified_edges")
    print("3. HAS_ARTICLE relationships between Laws and their Sections")
    print("4. CITATION relationships between Sections and target Laws")
    print("\nIf any of these components are missing, review the following scripts:")
    print("- ./scripts/DB_Population/nodes.py")
    print("- ./scripts/DB_Population/articles.py")
            
except Exception as e:
    print(f"Error in cleanup: {e}")